# Veri Toplama Pipeline v6

GitHub'dan Python projelerini toplayarak ham metrik dataseti oluşturur.
Filtreleme ve kategorizasyon **filter_categorize.ipynb** notebook'unda yapılır.

**Pipeline'da uygulanan tek filtreler:**
- Dil: Python
- Fork/archived değil
- MIN_SLOC = 10 (radon çalışsın diye)

**Dataset içeriği:**
- 22 statik kod metriği (radon)
- Proje bilgileri: star, contributor_count, project_age_days
- commit_count, n_authors, file_age_days, churn_total, avg_churn_per_commit, max_single_churn, recent_commits_90d (dosya bazında)
- bug_count + has_bug + bug_severity (git log'dan keyword mining)
- CodeBERT için ham kod (ilk 2500 karakter)
- label (commit_count > 1 → 1, değilse 0)

In [ ]:
!pip install requests radon pandas tqdm matplotlib

In [ ]:
import os
import re
import json
import time
import shutil
import subprocess
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from tqdm import tqdm
from datetime import datetime, timezone, timedelta

from radon.complexity import cc_visit
from radon.metrics import mi_visit, h_visit
from radon.raw import analyze

In [ ]:
# ============================================================
# AYARLAR
# ============================================================

GITHUB_TOKEN = input("GitHub Token: ").strip()

WORK_DIR = Path(r"C:\Users\bm123\Documents\MetricHunter")
CLONE_DIR = WORK_DIR / "repos"
OUTPUT_DIR = WORK_DIR / "output"
CLONE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

# Hedef proje sayisi — filtreleme ayri notebook'ta yapilacak
TARGET_PROJECT_COUNT = 500

# Metrik filtreleri (pipeline seviyesinde sadece bunlar kalir)
MIN_SLOC = 10           # radon calissin diye minimum
MAX_FILE_SIZE = 500_000

# Dosya filtreleme (test, init, migration vb. — her zaman atlanir)
SKIP_PATTERNS = [
    r"__init__\.py$",
    r"setup\.py$",
    r"conftest\.py$",
    r"manage\.py$",
    r"/migrations?/",
    r"/tests?/",
    r"test_[^/]+\.py$",
    r"[^/]+_test\.py$",
    r"/docs?/",
    r"/examples?/",
    r"/vendor/",
    r"conf\.py$",
    r"__main__\.py$",
    r"/\.",
]
SKIP_REGEX = re.compile("|".join(SKIP_PATTERNS))

# Bug keyword'leri
BUG_KEYWORDS = re.compile(
    r'(?:'
    r'\b(fix(es|ed|ing)?|bug(s|gy)?|defect(s|ive)?|fault(s|y)?)\b'
    r'|\b(error(s)?|patch(es|ed|ing)?|resolv(e|es|ed|ing)?)\b'
    r'|\b(crash(es|ed|ing)?|regression(s)?)\b'
    r'|\b(hotfix|bugfix|workaround|repair(ed|ing)?|typo(s)?)\b'
    r'|\b(broken|failure(s)?|fail(s|ed|ing)?)\b'
    r'|\b(null.?pointer|npe|exception|traceback|segfault|overflow)\b'
    r'|^fix[\(:]'
    r'|\b(fix(?:es|ed)?|close[sd]?|resolve[sd]?)\s*#\d+'
    r')',
    re.IGNORECASE | re.MULTILINE
)

print(f"Calisma dizini: {WORK_DIR.resolve()}")
print(f"Hedef proje sayisi: {TARGET_PROJECT_COUNT}")

In [ ]:
# Token kontrolu
resp = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
rate = resp.json()["rate"]
print(f"API Rate Limit: {rate['remaining']}/{rate['limit']}")

if rate['limit'] < 1000:
    print("Token taninmadi! Kontrol et.")
else:
    print("Token gecerli.")

---
## 1. Proje Kesfi

In [ ]:
def get_contributor_count(full_name):
    """GitHub API ile contributor sayisini al."""
    try:
        # per_page=1 ile sadece header'dan toplam sayiyi al
        resp = requests.get(
            f"https://api.github.com/repos/{full_name}/contributors",
            headers=HEADERS,
            params={"per_page": 1, "anon": "true"}
        )
        if resp.status_code != 200:
            return None
        
        # Link header'dan son sayfa numarasini al
        link_header = resp.headers.get("Link", "")
        if 'rel="last"' in link_header:
            # ...&page=42>; rel="last" formatindan sayiyi cek
            import re as _re
            match = _re.search(r'[&?]page=(\d+)>; rel="last"', link_header)
            if match:
                return int(match.group(1))
        
        # Link header yoksa sonuc tek sayfaya sigiyor
        return len(resp.json())
    
    except Exception:
        return None


def calculate_project_age_days(created_at_str):
    """Projenin olusturulma tarihinden bu yana gecen gun sayisi."""
    try:
        created = datetime.fromisoformat(created_at_str.replace("Z", "+00:00"))
        now = datetime.now(timezone.utc)
        return (now - created).days
    except Exception:
        return None

In [ ]:
def search_github_repos(max_projects=500):
    """GitHub API ile Python projelerini ara. Genis yelpazeye acildi."""
    all_repos = []
    page = 1

    # Sadece dil + fork/archived filtresi — geri kalanlar filter_categorize'da
    query = (
        "language:python "
        "fork:false "
        "archived:false"
    )

    print(f"Sorgu: {query}")
    print(f"Hedef: {max_projects} proje\n")

    while len(all_repos) < max_projects:
        resp = requests.get(
            "https://api.github.com/search/repositories",
            headers=HEADERS,
            params={"q": query, "sort": "stars", "order": "desc",
                    "per_page": 30, "page": page}
        )

        if resp.status_code != 200:
            print(f"Hata: {resp.status_code} - {resp.json().get('message', '')}")
            if resp.status_code == 403:
                reset_time = int(resp.headers.get('X-RateLimit-Reset', 0))
                wait = max(reset_time - time.time(), 10)
                print(f"{wait:.0f} saniye bekleniyor...")
                time.sleep(wait + 1)
                continue
            break

        items = resp.json().get("items", [])
        if not items:
            print("Baska sonuc yok.")
            break

        for repo in items:
            all_repos.append({
                "name": repo["name"],
                "full_name": repo["full_name"],
                "clone_url": repo["clone_url"],
                "html_url": repo["html_url"],
                "stars": repo["stargazers_count"],
                "forks": repo["forks_count"],
                "size_kb": repo["size"],
                "language": repo["language"],
                "created_at": repo["created_at"],
                "updated_at": repo["updated_at"],
                "description": repo.get("description", ""),
                "topics": repo.get("topics", []),
                "default_branch": repo.get("default_branch", "main"),
                "open_issues": repo["open_issues_count"],
                "license": repo["license"]["spdx_id"] if repo.get("license") else None
            })

        print(f"  Sayfa {page}: {len(items)} proje (Toplam: {len(all_repos)})")
        page += 1
        time.sleep(2)

        if len(all_repos) >= max_projects:
            break

    all_repos = all_repos[:max_projects]
    print(f"\n{len(all_repos)} proje bulundu.")
    return all_repos

In [ ]:
repos_file = OUTPUT_DIR / "discovered_repos.json"

if repos_file.exists():
    with open(repos_file, "r", encoding="utf-8") as f:
        repos_raw = json.load(f)
    print(f"Onceki kesif yuklendi: {len(repos_raw)} proje")
else:
    repos_raw = search_github_repos(TARGET_PROJECT_COUNT)
    with open(repos_file, "w", encoding="utf-8") as f:
        json.dump(repos_raw, f, indent=2, ensure_ascii=False)
    print(f"Kaydedildi: {repos_file}")

print(f"Toplam bulunan: {len(repos_raw)} proje")

In [ ]:
# Contributor sayisini al 
enriched_file = OUTPUT_DIR / "enriched_repos.json"

if enriched_file.exists():
    with open(enriched_file, "r", encoding="utf-8") as f:
        repos = json.load(f)
    print(f"Onceki liste yuklendi: {len(repos)} proje")
else:
    repos = []
    api_errors = 0
    
    for repo in tqdm(repos_raw, desc="Contributor kontrolu"):
        age_days = calculate_project_age_days(repo["created_at"])
        contrib_count = get_contributor_count(repo["full_name"])
        time.sleep(0.5)
        
        if contrib_count is None:
            api_errors += 1
            contrib_count = 0  # API hatasi olsa bile projeyi atma
        
        repo["contributor_count"] = contrib_count
        repo["project_age_days"] = age_days
        repos.append(repo)
    
    with open(enriched_file, "w", encoding="utf-8") as f:
        json.dump(repos, f, indent=2, ensure_ascii=False)
    
    print(f"\nToplam: {len(repos)} proje (API hatasi: {api_errors})")

df_repos = pd.DataFrame(repos)
print(f"\nProje sayisi: {len(df_repos)}")
if len(df_repos) > 0:
    print(f"Star araligi: {df_repos['stars'].min()} - {df_repos['stars'].max()}")
    print(f"Contributor araligi: {df_repos['contributor_count'].min()} - {df_repos['contributor_count'].max()}")
    print(f"Yas araligi: {df_repos['project_age_days'].min()} - {df_repos['project_age_days'].max()} gun")

---
## 2. Yardimci Fonksiyonlar

In [ ]:
def clone_repo(repo_info, base_dir):
    """Normal klon: dosyalar + gecmis tamamen diske iner."""
    repo_name = repo_info["full_name"].replace("/", "_")
    repo_path = base_dir / repo_name
    
    if repo_path.exists():
        return repo_path, "zaten_var"
    
    try:
        cmd = [
            "git", "clone",
            "--single-branch",
            repo_info["clone_url"],
            str(repo_path)
        ]
        result = subprocess.run(
            cmd, capture_output=True, text=True, timeout=600,
            encoding="utf-8", errors="replace"
        )
        if result.returncode != 0:
            return None, f"hata: {result.stderr[:200]}"
        return repo_path, "basarili"
    except subprocess.TimeoutExpired:
        if repo_path.exists():
            shutil.rmtree(repo_path, ignore_errors=True)
        return None, "timeout"
    except Exception as e:
        if repo_path.exists():
            shutil.rmtree(repo_path, ignore_errors=True)
        return None, f"hata: {str(e)[:200]}"


def get_head_python_files(repo_path):
    """HEAD'deki tum Python dosyalarini listele."""
    try:
        result = subprocess.run(
            ["git", "ls-tree", "-r", "--name-only", "HEAD"],
            cwd=str(repo_path), capture_output=True, text=True, timeout=30,
            encoding="utf-8", errors="replace"
        )
        if result.returncode != 0:
            return []
        files = result.stdout.strip().split("\n")
        return [f for f in files if f.endswith(".py") and f.strip()]
    except Exception:
        return []


def should_skip_file(file_path):
    """Test, init, migration gibi dosyalari filtrele."""
    return bool(SKIP_REGEX.search(file_path))


def get_bulk_git_stats(repo_path, head_files):
    """
    Tek git log komutuyla tum dosya bazli git metriklerini cikar:
    - commit_count: toplam commit sayisi
    - bug_count: bug-fix commit sayisi
    - n_authors: farkli yazar sayisi
    - file_age_days: ilk committen bugune gun
    - churn_total: toplam eklenen + silinen satir
    - avg_churn_per_commit: commit basina ortalama churn
    - max_single_churn: tek committeki en buyuk degisiklik
    - recent_commits_90d: son 90 gundeki commit sayisi
    """
    try:
        result = subprocess.run(
            ["git", "log", "--format=COMMIT|%H|%ae|%at|%s", "--numstat", "HEAD"],
            cwd=str(repo_path), capture_output=True, text=True,
            timeout=300, encoding="utf-8", errors="replace"
        )
        if result.returncode != 0:
            return {}

        head_set = set(head_files)
        now = datetime.now(timezone.utc)
        cutoff_90d = (now - timedelta(days=90)).timestamp()

        # Her dosya icin istatistik toplama
        file_stats = {}
        for f in head_files:
            file_stats[f] = {
                "commit_count": 0,
                "bug_count": 0,
                "authors": set(),
                "timestamps": [],
                "churns": [],
            }

        current_is_bug = False
        current_timestamp = 0
        current_author = ""

        for line in result.stdout.split("\n"):
            line = line.strip()
            if not line:
                continue

            if line.startswith("COMMIT|"):
                parts = line.split("|", 4)
                if len(parts) >= 5:
                    current_author = parts[2]
                    try:
                        current_timestamp = int(parts[3])
                    except ValueError:
                        current_timestamp = 0
                    msg = parts[4]
                    current_is_bug = bool(BUG_KEYWORDS.search(msg))
            else:
                # numstat satiri: "eklenen\tsilinen\tdosya_yolu"
                parts = line.split("\t")
                if len(parts) == 3:
                    added_str, deleted_str, fpath = parts
                    if fpath not in head_set:
                        continue

                    stats = file_stats.get(fpath)
                    if stats is None:
                        continue

                    stats["commit_count"] += 1
                    if current_is_bug:
                        stats["bug_count"] += 1
                    stats["authors"].add(current_author)
                    stats["timestamps"].append(current_timestamp)

                    # churn hesapla (binary dosyalarda "-" olur)
                    try:
                        added = int(added_str)
                        deleted = int(deleted_str)
                        stats["churns"].append(added + deleted)
                    except ValueError:
                        pass

        # Sonuclari hesapla
        results = {}
        for fpath, stats in file_stats.items():
            if stats["commit_count"] == 0:
                continue

            churns = stats["churns"]
            timestamps = stats["timestamps"]

            oldest_ts = min(timestamps) if timestamps else now.timestamp()
            file_age = max((now.timestamp() - oldest_ts) / 86400, 0)
            recent = sum(1 for ts in timestamps if ts >= cutoff_90d)

            results[fpath] = {
                "commit_count": stats["commit_count"],
                "bug_count": stats["bug_count"],
                "n_authors": len(stats["authors"]),
                "file_age_days": round(file_age, 1),
                "churn_total": sum(churns) if churns else 0,
                "avg_churn_per_commit": round(sum(churns) / len(churns), 1) if churns else 0,
                "max_single_churn": max(churns) if churns else 0,
                "recent_commits_90d": recent,
            }

        return results
    except Exception:
        return {}

def get_file_content(repo_path, file_path):
    """Dosyayi diskten oku."""
    try:
        full_path = repo_path / file_path
        if full_path.exists():
            return full_path.read_text(encoding="utf-8", errors="replace")
        return None
    except Exception:
        return None


def truncate_code_for_bert(source_code, max_chars=2500):
    """Kodun orijinal formatini bozmadan karakter bazli kirp."""
    if len(source_code) > max_chars:
        return source_code[:max_chars]
    return source_code


def calculate_metrics(source_code):
    """Radon ile statik metrikleri hesapla."""
    if len(source_code) > MAX_FILE_SIZE:
        return None
    
    metrics = {}
    
    try:
        raw = analyze(source_code)
        metrics["loc"] = raw.loc
        metrics["lloc"] = raw.lloc
        metrics["sloc"] = raw.sloc
        metrics["comments"] = raw.comments
        metrics["multi"] = raw.multi
        metrics["blank"] = raw.blank
        metrics["single_comments"] = raw.single_comments
    except Exception:
        return None
    
    try:
        cc_results = cc_visit(source_code)
        if cc_results:
            complexities = [block.complexity for block in cc_results]
            metrics["cc_mean"] = sum(complexities) / len(complexities)
            metrics["cc_max"] = max(complexities)
            metrics["cc_total"] = sum(complexities)
            metrics["num_functions"] = len(cc_results)
        else:
            metrics["cc_mean"] = 0
            metrics["cc_max"] = 0
            metrics["cc_total"] = 0
            metrics["num_functions"] = 0
    except Exception:
        metrics["cc_mean"] = None
        metrics["cc_max"] = None
        metrics["cc_total"] = None
        metrics["num_functions"] = None
    
    try:
        halstead = h_visit(source_code)
        if halstead.total and hasattr(halstead.total, 'volume'):
            h = halstead.total
            metrics["h_vocabulary"] = h.vocabulary
            metrics["h_length"] = h.length
            metrics["h_volume"] = h.volume
            metrics["h_difficulty"] = h.difficulty
            metrics["h_effort"] = h.effort
            metrics["h_bugs"] = h.bugs
            metrics["h_time"] = h.time
            metrics["h_calculated_length"] = h.calculated_length
        else:
            for k in ["h_vocabulary", "h_length", "h_volume", "h_difficulty",
                      "h_effort", "h_bugs", "h_time", "h_calculated_length"]:
                metrics[k] = 0
    except Exception:
        for k in ["h_vocabulary", "h_length", "h_volume", "h_difficulty",
                  "h_effort", "h_bugs", "h_time", "h_calculated_length"]:
            metrics[k] = None
    
    try:
        mi = mi_visit(source_code, multi=True)
        metrics["maintainability_index"] = mi
    except Exception:
        metrics["maintainability_index"] = None
    
    return metrics

---
## 3. Klonlama Sonuc Takibi

Klonlama artik ana pipeline dongusunde yapiliyor (her proje: klonla -> isle -> sil).
Bu hucre sadece onceki kosulardan kalan klon sonuclarini yukler.

In [ ]:
clone_file = OUTPUT_DIR / "clone_results.json"

if clone_file.exists():
    with open(clone_file, "r", encoding="utf-8") as f:
        clone_results = json.load(f)
    print(f"Onceki klon sonuclari yuklendi: {len(clone_results)} kayit")
else:
    clone_results = []
    print("Klonlama+isleme dongusu icinde olusturulacak.")

---
## 4. Ana Pipeline: Klonla + Isle + Sil

Her proje icin:
1. Repoyu klonla (CLONE_DIR altina)
2. HEAD'deki Python dosyalarini bul
3. Dosya filtreleme (test, init vb.)
4. Bulk git stats (commit, bug, churn, author, age)
5. Statik metrik hesaplama + CodeBERT token
6. Proje bilgilerini ekle (star, contributor, age)
7. **Repoyu disk'ten sil** (disk yerini bosalt)

In [ ]:
# Proje bilgilerine hizli erisim icin dict olustur
repo_info_map = {r["full_name"]: r for r in repos}

# Checkpoint
checkpoint_file = OUTPUT_DIR / "metrics_checkpoint.csv"
progress_file = OUTPUT_DIR / "processed_projects.json"

if checkpoint_file.exists():
    df_checkpoint = pd.read_csv(checkpoint_file)
    all_records = df_checkpoint.to_dict('records')
    print(f"Checkpoint bulundu: {len(all_records)} dosya")
else:
    all_records = []
    print("Sifirdan baslaniyor...")

if progress_file.exists():
    with open(progress_file, "r") as f:
        processed_projects = set(json.load(f))
    print(f"Daha once islenmis proje: {len(processed_projects)}")
else:
    processed_projects = set()

errors = []
project_stats = []
save_every_projects = 5

In [ ]:
projects_since_save = 0

for repo in tqdm(repos, desc="Pipeline"):
    repo_name = repo["full_name"]

    if repo_name in processed_projects:
        continue

    # 1. Klonla
    repo_path, status = clone_repo(repo, CLONE_DIR)
    clone_results.append({
        "full_name": repo_name,
        "status": status
    })

    if repo_path is None:
        tqdm.write(f"  {repo_name}: klon hatasi - {status}")
        processed_projects.add(repo_name)
        project_stats.append({"project": repo_name, "status": f"klon_hatasi:{status}", "files": 0})
        continue

    try:
        info = repo_info_map.get(repo_name, {})
        stars = info.get("stars", 0)
        contributor_count = info.get("contributor_count", 0)
        project_age_days = info.get("project_age_days", 0)

        # 2. HEAD dosyalari
        head_files = get_head_python_files(repo_path)
        if not head_files:
            processed_projects.add(repo_name)
            project_stats.append({"project": repo_name, "status": "dosya_yok", "files": 0})
            continue

        # 3. Dosya filtreleme
        filtered_files = [f for f in head_files if not should_skip_file(f)]
        if not filtered_files:
            processed_projects.add(repo_name)
            project_stats.append({"project": repo_name, "status": "filtre_sonrasi_bos", "files": 0})
            continue

        # 4. Git metrikleri (tek komutla)
        git_stats = get_bulk_git_stats(repo_path, filtered_files)

        # 5. Metrik hesaplama
        file_count = 0
        for file_path in filtered_files:
            fstats = git_stats.get(file_path)
            if fstats is None or fstats["commit_count"] <= 0:
                continue

            content = get_file_content(repo_path, file_path)
            if content is None or len(content.strip()) == 0:
                errors.append({"project": repo_name, "file": file_path, "error": "icerik_yok"})
                continue

            metrics = calculate_metrics(content)
            if metrics is None:
                errors.append({"project": repo_name, "file": file_path, "error": "metrik_hatasi"})
                continue

            if metrics.get("sloc", 0) < MIN_SLOC:
                errors.append({"project": repo_name, "file": file_path, "error": "cok_kisa"})
                continue

            code_tokens = truncate_code_for_bert(content)

            record = {
                "project": repo_name,
                "file_path": file_path,
                # Proje bilgileri
                "stars": stars,
                "contributor_count": contributor_count,
                "project_age_days": project_age_days,
                # Git metrikleri
                **fstats,
                # Statik metrikler
                **metrics,
                # Turetilmis metrikler
                "complexity_density": (metrics["cc_total"] or 0) / max(metrics["loc"], 1),
                "comment_per_function": (metrics["comments"] or 0) / max(metrics["num_functions"] or 0, 1),
                "avg_function_length": (metrics["sloc"] or 0) / max(metrics["num_functions"] or 0, 1),
                "effort_per_line": (metrics["h_effort"] or 0) / max(metrics["loc"], 1),
                # CodeBERT
                "code_tokens": code_tokens,
            }
            all_records.append(record)
            file_count += 1

        processed_projects.add(repo_name)
        project_stats.append({"project": repo_name, "status": "basarili", "files": file_count})
        total_bugs = sum(s["bug_count"] for s in git_stats.values())
        tqdm.write(f"  {repo_name}: {file_count} dosya (git_stats: {len(git_stats)}, bugs: {total_bugs})")

    finally:
        # 7. Repoyu disk'ten sil (basarili da olsa, hata da olsa)
        if repo_path and repo_path.exists():
            shutil.rmtree(repo_path, ignore_errors=True)

    # Checkpoint
    projects_since_save += 1
    if projects_since_save >= save_every_projects:
        pd.DataFrame(all_records).to_csv(checkpoint_file, index=False)
        with open(progress_file, "w") as f:
            json.dump(list(processed_projects), f)
        with open(clone_file, "w", encoding="utf-8") as f:
            json.dump(clone_results, f, indent=2, ensure_ascii=False)
        tqdm.write(f"  Checkpoint: {len(all_records)} dosya, {len(processed_projects)} proje")
        projects_since_save = 0

# Final checkpoint
pd.DataFrame(all_records).to_csv(checkpoint_file, index=False)
with open(progress_file, "w") as f:
    json.dump(list(processed_projects), f)
with open(clone_file, "w", encoding="utf-8") as f:
    json.dump(clone_results, f, indent=2, ensure_ascii=False)
pd.DataFrame(project_stats).to_csv(OUTPUT_DIR / "project_stats.csv", index=False)

print("\n" + "=" * 60)
print("PIPELINE TAMAMLANDI")
print("=" * 60)
print(f"Islenen proje: {len(processed_projects)}")
print(f"Toplam dosya: {len(all_records)}")
print(f"Hata/Atlanan: {len(errors)}")

if errors:
    df_errors = pd.DataFrame(errors)
    print("\nHata dagilimi:")
    print(df_errors["error"].value_counts())

---
## 5. Temizlik ve Label Olusturma

In [ ]:
df = pd.DataFrame(all_records)

# Label'lar
median_commits = df["commit_count"].median()
df["label"] = (df["commit_count"] > median_commits).astype(int)
print(f"Commit median esigi: {median_commits}")
print(f"Label dengesi: 0={( df['label']==0).sum()} ({(df['label']==0).mean():.1%}), 1={(df['label']==1).sum()} ({(df['label']==1).mean():.1%})")
df["has_bug"] = (df["bug_count"] > 0).astype(int)
df["bug_severity"] = pd.cut(df["bug_count"], bins=[-1, 0, 2, float("inf")], labels=[0, 1, 2]).astype(int)
# 0: bug yok, 1: az buglu (1-2), 2: cok buglu (3+)

# Turetilmis metrikler
df["comment_ratio"] = df["comments"] / df["loc"].replace(0, 1)
df["doc_ratio"] = df["multi"] / df["loc"].replace(0, 1)

# Metrik sutunlarinda NaN temizle
metric_cols = [
    "loc", "lloc", "sloc", "comments", "multi", "blank", "single_comments",
    "cc_mean", "cc_max", "cc_total", "num_functions",
    "h_vocabulary", "h_length", "h_volume", "h_difficulty",
    "h_effort", "h_bugs", "h_time", "h_calculated_length",
    "maintainability_index",
    "comment_ratio", "doc_ratio"
]
df = df.dropna(subset=metric_cols)

# MI = 0
mi_zero = (df["maintainability_index"] == 0).sum()
if mi_zero > 0:
    print(f"maintainability_index=0: {mi_zero} dosya eleniyor")
    df = df[df["maintainability_index"] > 0]

print(f"Temiz veri: {len(df)} dosya, {df['project'].nunique()} proje")

---
## 6. Dataset Ozeti ve Gorsellestirme

In [ ]:
print("=" * 60)
print("DATASET OZETI")
print("=" * 60)
print(f"Dosya sayisi: {len(df)}")
print(f"Proje sayisi: {df['project'].nunique()}")
print(f"")
print(f"--- Sinif Dagilimi (label: commit) ---")
print(f"Sinif 0 (tek commit): {(df['label']==0).sum()} ({(df['label']==0).mean():.1%})")
print(f"Sinif 1 (coklu commit): {(df['label']==1).sum()} ({(df['label']==1).mean():.1%})")
print(f"")
print(f"--- Bug Dagilimi ---")
print(f"Bug olan dosya: {(df['has_bug']==1).sum()} ({(df['has_bug']==1).mean():.1%})")
print(f"Bug olmayan dosya: {(df['has_bug']==0).sum()} ({(df['has_bug']==0).mean():.1%})")
print(f"Toplam bug sayisi: {df['bug_count'].sum()}")
print(f"")
print(f"--- Bug Severity Dagilimi ---")
for sev, label_name in [(0, "Bug yok"), (1, "Az buglu (1-2)"), (2, "Cok buglu (3+)")]:
    n = (df["bug_severity"] == sev).sum()
    print(f"  {label_name}: {n} ({n/len(df):.1%})")
print(f"")
print(f"--- Proje Bilgileri ---")
print(f"Star: min={df.groupby('project')['stars'].first().min()}, max={df.groupby('project')['stars'].first().max()}")
print(f"Contributor: min={df.groupby('project')['contributor_count'].first().min()}, max={df.groupby('project')['contributor_count'].first().max()}")
print(f"Yas (gun): min={df.groupby('project')['project_age_days'].first().min()}, max={df.groupby('project')['project_age_days'].first().max()}")
print(f"")
pp = df.groupby("project").size()
print(f"--- Proje Basina Dosya ---")
print(f"min={pp.min()}, max={pp.max()}, mean={pp.mean():.0f}, median={pp.median():.0f}")

# Domine eden proje kontrolu
per_sorted = pp.sort_values(ascending=False)
total = per_sorted.sum()
cumsum = per_sorted.cumsum()
print(f"\n--- Proje Agirligi ---")
print(f"En buyuk proje: {per_sorted.index[0]} ({per_sorted.iloc[0]} dosya, {per_sorted.iloc[0]/total*100:.1f}%)")
top5 = per_sorted.head(5)
print(f"Ilk 5 proje: {top5.sum()} dosya ({top5.sum()/total*100:.1f}%)")
for threshold in [25, 50, 75]:
    n = (cumsum <= total * threshold / 100).sum()
    print(f"  Verinin %{threshold}'i ilk {n} projeden geliyor")

In [ ]:
# Gorsellestirme
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Commit sinif dagilimi
counts = df["label"].value_counts().sort_index()
labels = ["Tek Commit (0)", "Coklu Commit (1)"]
colors = ["#e74c3c", "#2ecc71"]
bars = axes[0].bar(labels, counts.values, color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, counts.values):
    pct = val / counts.sum() * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.02,
                 f"{val}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title("Commit Sinif Dagilimi", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Dosya Sayisi")
axes[0].set_ylim(0, counts.max() * 1.25)

# 2. Bug dagilimi
bug_counts = df["has_bug"].value_counts().sort_index()
labels2 = ["Bug Yok (0)", "Bug Var (1)"]
colors2 = ["#3498db", "#e67e22"]
bars2 = axes[1].bar(labels2, bug_counts.values, color=colors2, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars2, bug_counts.values):
    pct = val / bug_counts.sum() * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + bug_counts.max()*0.02,
                 f"{val}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title("Bug Dagilimi", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Dosya Sayisi")
axes[1].set_ylim(0, bug_counts.max() * 1.25)

# 3. Proje basina dosya dagilimi
axes[2].hist(pp.values, bins=30, color="#9b59b6", edgecolor="white")
axes[2].axvline(pp.mean(), color='red', linestyle='--', label=f'Ortalama: {pp.mean():.0f}')
axes[2].axvline(pp.median(), color='orange', linestyle='--', label=f'Medyan: {pp.median():.0f}')
axes[2].set_title("Proje Basina Dosya Dagilimi", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Dosya Sayisi")
axes[2].set_ylabel("Proje Sayisi")
axes[2].legend()

plt.suptitle(f"Dataset Ozeti ({len(df)} dosya, {df['project'].nunique()} proje)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dataset_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Grafik kaydedildi: {OUTPUT_DIR / 'dataset_summary.png'}")

---
## 7. Kaydetme

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Tam dataset (her sey dahil)
full_path = OUTPUT_DIR / f"dataset_full_{timestamp}.csv"
df.to_csv(full_path, index=False)
print(f"Tam dataset: {full_path.name} ({len(df)} satir)")

# Model dataset (metrikler + proje bilgileri + commit_count + bug + label)
model_cols = (
    ["project", "file_path"]
    + ["stars", "contributor_count", "project_age_days"]
    + ["commit_count", "bug_count", "n_authors", "file_age_days", "churn_total",
       "avg_churn_per_commit", "max_single_churn", "recent_commits_90d"]
    + metric_cols
    + ["complexity_density", "comment_per_function", "avg_function_length", "effort_per_line"]
    + ["label", "has_bug", "bug_severity"]
)
model_path = OUTPUT_DIR / f"dataset_model_{timestamp}.csv"
df[model_cols].to_csv(model_path, index=False)
print(f"Model dataset: {model_path.name} ({len(df)} satir, {len(model_cols)} sutun)")

# CodeBERT dataset
bert_path = OUTPUT_DIR / f"dataset_codebert_{timestamp}.csv"
df[["project", "file_path", "code_tokens", "label", "has_bug", "bug_severity"]].to_csv(bert_path, index=False)
print(f"CodeBERT dataset: {bert_path.name}")

# Checkpoint temizle
if checkpoint_file.exists():
    checkpoint_file.unlink()
if progress_file.exists():
    progress_file.unlink()
print("\nCheckpoint dosyalari silindi.")

print(f"\nDataset basariyla olusturuldu!")
print(f"  {len(metric_cols)} metrik sutunu")
print(f"  + proje bilgileri (stars, contributor_count, project_age_days)")
print(f"  + commit_count, bug_count, has_bug")
print(f"  + CodeBERT icin ham kod (ilk 2500 karakter)")
print(f"  Konum: {OUTPUT_DIR}")

In [ ]:
# Metrik istatistikleri
print("Metrik istatistikleri:")
for col in metric_cols:
    print(f"  {col:35s} min={df[col].min():>10.2f}  max={df[col].max():>10.2f}  mean={df[col].mean():>10.2f}")

# Yeni git metrikleri
print("\nYeni git metrikleri:")
for col in ["n_authors", "file_age_days", "churn_total", "avg_churn_per_commit",
            "max_single_churn", "recent_commits_90d"]:
    if col in df.columns:
        print(f"  {col:35s} min={df[col].min():>10.2f}  max={df[col].max():>10.2f}  mean={df[col].mean():>10.2f}")

# Turetilmis metrikler
print("\nTuretilmis metrikler:")
for col in ["complexity_density", "comment_per_function", "avg_function_length", "effort_per_line"]:
    if col in df.columns:
        print(f"  {col:35s} min={df[col].min():>10.2f}  max={df[col].max():>10.2f}  mean={df[col].mean():>10.2f}")

# Bug severity dagilimi
if "bug_severity" in df.columns:
    print(f"\n--- Bug Severity Dagilimi ---")
    for sev, label_name in [(0, "Bug yok"), (1, "Az buglu (1-2)"), (2, "Cok buglu (3+)")]:
        n = (df["bug_severity"] == sev).sum()
        print(f"  {label_name}: {n} ({n/len(df):.1%})")

In [ ]:
# Sutun listesi
print("Dataset sutunlari:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

In [ ]:
df.head(10)